In [ ]:
import numpy as np
from NRE import nre
from sklearn.metrics import mean_squared_error, log_loss, zero_one_loss
import time
from itertools import product
import pandas as pd
from dataset import BootstrapSplitter, CallData, TrainTestSplitter

In [ ]:
def nre_experiment(num_splits = 10, max_depth = [3], dataset_name = None, random_state = 111, lambda_value_rule_weights=[1e-10],  
                   save_results = True, train_size = 500):
    dataset_name = dataset_name
    dataset = CallData().call(dataset_name = dataset_name, data_format = 'df')
    X = dataset.X
    y = dataset.y
    train_size = min(train_size, len(X))

    
    # all_rule_models = np.empty(shape=(num_splits, max_depth, max_leaf_nodes, min_samples_split-1), dtype=object)

    df = pd.DataFrame(columns=['rep no', 'max depth', 'test loss', 'train loss', 'test 01 loss', 'train 01 loss', 'num rules', 'rules complexity', 'model complexity', 'comp time (sec)'])
    elapsed_time = pd.DataFrame(columns=['dataset name', 'time (sec)'])
    split_ind = 0
    df_ind = 0
    print('regression type is '+dataset.learning_type)
    splitter = BootstrapSplitter(reps=num_splits, train_size=train_size, random_state=random_state, replace=True)
    start_time = time.time()
    for train_idx, test_idx in splitter.split(X):
        x_train = X.loc[train_idx]
        y_train = y.loc[train_idx]
        x_test = X.loc[test_idx]
        y_test = y.loc[test_idx]
        print(f'----train set sample index sum: {np.sum(train_idx)}----')
        for depth_ind, depth in enumerate(max_depth):
            print(f'----repetition number {split_ind+1}----')
            print(f'----max depth {depth}----')
            start_time_one_iteration = time.time()
            model_rule = nre.NREModel(max_tree_depth=depth, task=dataset.learning_type).fit(x_train,y_train)
            end_time_one_iteration = time.time()
    
            if dataset.learning_type == 'linreg':
                test_loss = mean_squared_error(y_test, model_rule.predict(x_test))
                train_loss = mean_squared_error(y_train, model_rule.predict(x_train))
                train_01loss = 0
                test_01loss = 0
            else:
                test_loss = log_loss(y_test, model_rule.predict_prob(x_test))
                train_loss = log_loss(y_train, model_rule.predict_prob(x_train))
                train_01loss = zero_one_loss(y_train, model_rule.predict(x_train))
                test_01loss = zero_one_loss(y_test, model_rule.predict(x_test))


            df.loc[df_ind, 'rep no'] = split_ind+1
            df.loc[df_ind,'max depth'] = depth
            df.loc[df_ind,'test loss'] = test_loss
            df.loc[df_ind,'train loss'] = train_loss
            df.loc[df_ind,'train 01 loss'] = train_01loss
            df.loc[df_ind,'test 01 loss'] = test_01loss
            df.loc[df_ind,'num rules'] = model_rule.num_of_rules()
            df.loc[df_ind,'rules complexity'] = model_rule.rules_complexity()
            df.loc[df_ind,'model complexity'] = model_rule.model_complexity()
            df.loc[df_ind, 'comp time (sec)'] = end_time_one_iteration - start_time_one_iteration

            df_ind += 1
        df.to_excel('experiments_results/3000_samples_15_rules/nre_3000_samples_15_rules_'+dataset_name+'.xlsx', sheet_name='Sheet1', index=False)
        split_ind += 1
    end_time = time.time()
    elapsed_time.loc[0,'time (sec)'] = end_time - start_time
    elapsed_time.loc[0,'dataset name'] = dataset_name
    if save_results is True:
        df.to_excel('experiments_results/3000_samples_15_rules/nre_3000_samples_15_rules_'+dataset_name+'.xlsx', sheet_name='Sheet1', index=False)
        with pd.ExcelWriter('experiments_results/3000_samples_15_rules/nre_3000_samples_15_rules_'+dataset_name+'.xlsx', engine='openpyxl', mode='a') as writer:
            elapsed_time.to_excel(writer, sheet_name='CalcTime', index=False)
        
    return df


In [ ]:
dataset_name_list = ['red_wine', 'diabetes', 'make_friedman1', 'used_car_price', 'liver', 'banknote', 'make_friedman2', 'make_friedman3', 'iris','breast_cancer', 'magic04','voice',  'adult',  'california_housing_price']
dataset_name_list = list(set(dataset_name_list) - set(['liver', 'iris', 'diabetes', 'voice', 'used_car_price', 'magic04', 'adult']))
num_splits = 15
train_size=3000
max_depth = [2, 4, 6]

In [ ]:
for ds in dataset_name_list:
    print(f'################################### {ds} ##################################')
    df_nre = methods.nre_experiment(num_splits=num_splits, max_depth=max_depth, dataset_name=ds, random_state=111, train_size=train_size)